# Exploratory Data Analysis (EDA)

This notebook explores the cleaned DataCo Supply Chain dataset to identify patterns and factors associated with delivery performance.

The main analytical focus of the project is late delivery risk. The EDA will use descriptive statistics, visualisations and statistical hypothesis testing to investigate relationships between delivery performance and factors such as shipping mode, product category, order value and time.

The findings from this analysis will be used to inform the machine-learning stage of the project, where a supervised classification model will be developed to predict late delivery risk.

In [2]:
%pip install plotly

  Using cached plotly-6.9.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached narwhals-2.25.0-py3-none-any.whl.metadata (15 kB)
Using cached plotly-6.9.0-py3-none-any.whl (9.9 MB)
Using cached narwhals-2.25.0-py3-none-any.whl (467 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go


## Load Raw Dataset

In [5]:
df_raw = pd.read_csv("../data/raw/DataCoSupplyChainDataset.csv", encoding='latin1')
df_raw.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


## Run the ETL pipeline

In [6]:
from src.etl_pipeline import etl_pipeline

df = etl_pipeline()
df.head()


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,product_price,product_status,shipping_date_dateorders,shipping_mode,shipping_delay,order_month,order_weekday,shipping_weekday,high_value_order,customer_full_name
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,2018-02-03 22:56:00,Standard Class,-1,1,Wednesday,Saturday,True,Cally Holloway
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,327.75,0,2018-01-18 12:27:00,Standard Class,1,1,Saturday,Thursday,True,Irene Luna
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,327.75,0,2018-01-17 12:06:00,Standard Class,0,1,Saturday,Wednesday,True,Gillian Maldonado
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,327.75,0,2018-01-16 11:45:00,Standard Class,-1,1,Saturday,Tuesday,True,Tana Tate
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,2018-01-15 11:24:00,Standard Class,-2,1,Saturday,Monday,True,Orli Hendricks


# dataset overview

In [7]:
print("shape of cleaned dataset:", df.shape)

shape of cleaned dataset: (180519, 59)


# preview first 10 rows

In [8]:
display(df.head(10))

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,product_price,product_status,shipping_date_dateorders,shipping_mode,shipping_delay,order_month,order_weekday,shipping_weekday,high_value_order,customer_full_name
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,2018-02-03 22:56:00,Standard Class,-1,1,Wednesday,Saturday,True,Cally Holloway
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,327.75,0,2018-01-18 12:27:00,Standard Class,1,1,Saturday,Thursday,True,Irene Luna
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,327.75,0,2018-01-17 12:06:00,Standard Class,0,1,Saturday,Wednesday,True,Gillian Maldonado
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,327.75,0,2018-01-16 11:45:00,Standard Class,-1,1,Saturday,Tuesday,True,Tana Tate
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,327.75,0,2018-01-15 11:24:00,Standard Class,-2,1,Saturday,Monday,True,Orli Hendricks
5,TRANSFER,6,4,18.580000,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,...,327.75,0,2018-01-19 11:03:00,Standard Class,2,1,Saturday,Friday,True,Kimberly Flowers
6,DEBIT,2,1,95.180000,288.420013,Late delivery,1,73,Sporting Goods,Caguas,...,327.75,0,2018-01-15 10:42:00,First Class,1,1,Saturday,Monday,True,Constance Terrell
7,TRANSFER,2,1,68.430000,285.140015,Late delivery,1,73,Sporting Goods,Miami,...,327.75,0,2018-01-15 10:21:00,First Class,1,1,Saturday,Monday,True,Erica Stevens
8,CASH,3,2,133.720001,278.589996,Late delivery,1,73,Sporting Goods,Caguas,...,327.75,0,2018-01-16 10:00:00,Second Class,1,1,Saturday,Tuesday,True,Nichole Olsen
9,CASH,2,1,132.149994,275.309998,Late delivery,1,73,Sporting Goods,San Ramon,...,327.75,0,2018-01-15 09:39:00,First Class,1,1,Saturday,Monday,True,Oprah Delacruz


# Data types

In [9]:
print("\nData types:")
display(df.dtypes)


Data types:


type                                      str
days_for_shipping_real                  int64
days_for_shipment_scheduled             int64
benefit_per_order                     float64
sales_per_customer                    float64
delivery_status                           str
late_delivery_risk                      int64
category_id                             int64
category_name                             str
customer_city                             str
customer_country                          str
customer_email                            str
customer_fname                            str
customer_id                             int64
customer_lname                            str
customer_password                         str
customer_segment                          str
customer_state                            str
customer_street                           str
customer_zipcode                      float64
department_id                           int64
department_name                   

# Missing values

In [10]:
print("\nmissing values:")
display(df.isna().sum())


missing values:


type                                0
days_for_shipping_real              0
days_for_shipment_scheduled         0
benefit_per_order                   0
sales_per_customer                  0
delivery_status                     0
late_delivery_risk                  0
category_id                         0
category_name                       0
customer_city                       0
customer_country                    0
customer_email                      0
customer_fname                      0
customer_id                         0
customer_lname                      0
customer_password                   0
customer_segment                    0
customer_state                      0
customer_street                     0
customer_zipcode                    0
department_id                       0
department_name                     0
latitude                            0
longitude                           0
market                              0
order_city                          0
order_countr

# Descriptive statistics

In [11]:
print("\ndescriptive statistics:")
display(df.describe())


descriptive statistics:


,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,late_delivery_risk,category_id,customer_id,customer_zipcode,department_id,latitude,...,order_profit_per_order,order_zipcode,product_card_id,product_category_id,product_description,product_price,product_status,shipping_date_dateorders,shipping_delay,order_month
count,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,...,180519.000000,24840.000000,180519.000000,180519.000000,0.0,180519.000000,180519.0,180519,180519.000000,180519.000000
mean,3.497654,2.931847,21.974989,183.107609,0.548291,31.851451,6691.379495,35920.529950,5.443460,29.719955,...,21.974989,55426.132327,692.509764,31.851451,NaN,141.232550,0.0,2016-06-16 05:45:23.202433,0.565807,6.235449
min,0.000000,0.000000,-4274.979980,7.490000,0.000000,2.000000,1.000000,0.000000,2.000000,-33.937553,...,-4274.979980,1040.000000,19.000000,2.000000,NaN,9.990000,0.0,2015-01-03 00:00:00,-2.000000,1.000000
25%,2.000000,2.000000,7.000000,104.379997,0.000000,18.000000,3258.500000,725.000000,4.000000,18.265432,...,7.000000,23464.000000,403.000000,18.000000,NaN,50.000000,0.0,2015-09-25 06:59:00,0.000000,3.000000
50%,3.000000,4.000000,31.520000,163.990005,1.000000,29.000000,6457.000000,19380.000000,5.000000,33.144863,...,31.520000,59405.000000,627.000000,29.000000,NaN,59.990002,0.0,2016-06-15 08:32:00,1.000000,6.000000
75%,5.000000,4.000000,64.800003,247.399994,1.000000,45.000000,9779.000000,78207.000000,7.000000,39.279617,...,64.800003,90008.000000,1004.000000,45.000000,NaN,199.990005,0.0,2017-03-04 21:29:00,1.000000,9.000000
max,6.000000,4.000000,911.799988,1939.989990,1.000000,76.000000,20757.000000,99205.000000,12.000000,48.781933,...,911.799988,99301.000000,1363.000000,76.000000,NaN,1999.989990,0.0,2018-02-06 22:14:00,4.000000,12.000000
std,1.623722,1.374449,104.433526,120.043670,0.497664,15.640064,4162.918106,37542.434755,1.629246,9.813646,...,104.433526,31919.279101,336.446807,15.640064,NaN,139.732492,0.0,NaN,1.490966,3.403571


# duplicate rows

In [12]:
print("\nnumber of duplicated rows:", df.duplicated().sum())


number of duplicated rows: 0


### Reflection

The dataset overview confirms that the cleaned DataCo Supply Chain Dataset is structured correctly after running the automated ETL pipeline. The dataset contains a mix of numerical and categorical variables, and the descriptive statistics highlight variation in shipping times, order values, and customer information.

This step provides a solid foundation for the next stages of the EDA.


## Core Statistical Concepts

Statistics are important in data analysis because they help us understand patterns, variation and relationships within a dataset.

### Mean

The mean is the average value of a dataset. It is calculated by adding all values together and dividing by the number of values.

In this project, the mean can be used to understand the average order value or average shipping time.

### Median

The median is the middle value when the data is arranged in order. It can be useful when a dataset contains unusually high or low values because it is less affected by extreme values than the mean.

### Mode

The mode is the value that appears most frequently in a dataset. It can be useful for identifying the most common category, shipping mode or other repeated value.

### Variance

Variance measures how spread out values are from the mean. A higher variance means that the values are more widely spread out.

### Standard Deviation

Standard deviation shows how much values typically vary from the mean. A smaller standard deviation means the values are closer to the average, while a larger standard deviation means there is more variation.

### Probability

Probability represents the likelihood of an event occurring. It ranges from 0 to 1, or 0% to 100%.

In this project, probability can be used to calculate the likelihood of an order being at risk of late delivery.

### Hypothesis Testing

Hypothesis testing is used to determine whether there is enough evidence to support a relationship or difference in a dataset.

In this project, chi-square tests are used to investigate whether variables such as order month and product category are associated with late delivery risk.

### Normal Distribution

A normal distribution is a common pattern where most values are close to the average, with fewer values occurring at the extremes.

Understanding distributions helps analysts identify patterns and unusual values within a dataset.

In [14]:
order_values = df['order_item_total']
print("Mean order value:", order_values.mean())
print("Median order value:", order_values.median())
print("Mode order value:", order_values.mode()[0])
print("Variance:", order_values.var())
print("Standard deviation:", order_values.std())

Mean order value: 183.10760850778374
Median order value: 163.9900055
Mode order value: 122.8399963
Variance: 14410.482713821504
Standard deviation: 120.04367002812562


# probability example

In [53]:
late_prob = df["late_delivery_risk"].mean()

print("Probability of late delivery risk:", late_prob)
print(f"Percentage of orders at risk of late delivery: {late_prob * 100:.2f}%")

Probability of late delivery risk: 0.5482913155955883
Percentage of orders at risk of late delivery: 54.83%


Probability of late delivery risk: 0.548...
Percentage of orders at risk of late delivery: 54.83%

More than half of all deliveries risk missing the scheduled shipping window.

Late delivery is a major operational problem.

## Late Delivery Risk

The main focus of this analysis is late delivery. The `late_delivery_risk` variable indicates whether an order is classified as being at risk of late delivery.

The distribution of this variable is examined to understand the proportion of late and on-time orders before investigating factors that may be associated with delivery performance.

In [40]:
df["late_delivery_risk"].value_counts(normalize=True) * 100

late_delivery_risk
1    54.829132
0    45.170868
Name: proportion, dtype: float64

### Interpretation

The target variable shows that 54.83% of orders are classified as having a late delivery risk, while 45.17% are not.

The classes are reasonably balanced, although there are slightly more orders classified as being at risk of late delivery. This distribution is suitable for further statistical analysis and supervised machine-learning classification.

## Hypothesis 1: Shipping mode affects the likelihood of late delivery

This hypothesis investigates whether certain shipping modes are more prone to late delivery.  
To test this, we group orders by `shipping_mode` and calculate the percentage of late deliveries for each mode.


In [20]:
# Calculate late delivery percentage per shipping mode
shipping_mode_late = (
    df.groupby('shipping_mode')['shipping_delay']
    .apply(lambda x: (x > 0).mean() * 100)
    .reset_index(name='late_delivery_percentage')
)

shipping_mode_late


,shipping_mode,late_delivery_percentage
0,First Class,100.000000
1,Same Day,47.827873
2,Second Class,79.730804
3,Standard Class,39.768171


first class shipping being 100% late seems like an anomaly to me so i check the value counts to make sure all the data for first class shipping mode is actually late.

In [21]:
df[df['shipping_mode'] == 'First Class']['shipping_delay'].value_counts().sort_index()

shipping_delay
1    27814
Name: count, dtype: int64

In [22]:
df['shipping_mode'].value_counts()


shipping_mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

In [23]:
fig = px.bar(
    shipping_mode_late,
    x='shipping_mode',
    y='late_delivery_percentage',
    title='Late Delivery Percentage by Shipping Mode',
    labels={'late_delivery_percentage': 'Late Delivery (%)'},
    color='late_delivery_percentage'
)

fig.show()

### Interpretation

The results show a clear variation in late delivery rates across shipping modes, supporting the hypothesis.

- **First Class (100%)**  
  All First Class orders in the dataset are marked as late.  
  This is not due to a small sample size (≈27,000 rows).  
  Instead, it suggests a systemic issue such as:
  - extremely strict delivery expectations (e.g., 1‑day delivery windows),
  - timestamp inconsistencies in the synthetic dataset,
  - or First Class being used for urgent orders that were already behind schedule.

- **Second Class (79.73%)**  
  A very high late delivery rate, indicating slower carriers or longer routes.

- **Same Day (47.83%)**  
  Nearly half of Same Day deliveries fail to meet the promised timeframe, suggesting capacity or scheduling constraints.

- **Standard Class (39.77%)**  
  The most reliable mode, with the lowest late delivery percentage.

## Hypothesis 2: Orders placed during different months have different late delivery rates

This hypothesis looks at whether the month an order was placed affects the chance of late delivery.

To investigate this, I will:

- Group orders by month.
- Calculate the percentage of late deliveries for each month.
- Compare the results using a bar chart.
- Use a chi-square test to see if there is a significant relationship between order month and late delivery risk.

**Null hypothesis (H₀):** There is no relationship between order month and late delivery risk.

**Alternative hypothesis (H₁):** There is a relationship between order month and late delivery risk.

In [41]:
monthly_late = (
    df.groupby("order_month")["late_delivery_risk"]
      .mean()
      .mul(100)
      .reset_index(name="late_delivery_percentage")
)

monthly_late

,order_month,late_delivery_percentage
0,1,54.658212
1,2,54.552963
2,3,55.254727
3,4,54.279236
4,5,54.738358
5,6,54.825286
6,7,54.057279
7,8,55.794369
8,9,55.297308
9,10,54.596681


In [42]:
fig = px.bar(
    monthly_late,
    x="order_month",
    y="late_delivery_percentage",
    title="Late Delivery Risk by Order Month",
    labels={
        "order_month": "Order Month",
        "late_delivery_percentage": "Late Delivery Risk (%)"
    }
)

fig.show()

In [44]:
!pip install scipy

   ---------------------------------------- 0.0/36.7 MB ? eta -:--:--
    --------------------------------------- 0.8/36.7 MB 4.8 MB/s eta 0:00:08
   -- ------------------------------------- 1.8/36.7 MB 5.0 MB/s eta 0:00:07
   --- ------------------------------------ 2.9/36.7 MB 4.9 MB/s eta 0:00:07
   ---- ----------------------------------- 4.2/36.7 MB 5.2 MB/s eta 0:00:07
   ------ --------------------------------- 5.5/36.7 MB 5.4 MB/s eta 0:00:06
   ------- -------------------------------- 6.8/36.7 MB 5.4 MB/s eta 0:00:06
   -------- ------------------------------- 7.9/36.7 MB 5.5 MB/s eta 0:00:06
   ---------- ----------------------------- 9.4/36.7 MB 5.6 MB/s eta 0:00:05
   ----------- ---------------------------- 10.5/36.7 MB 5.6 MB/s eta 0:00:05
   ------------ --------------------------- 11.8/36.7 MB 5.6 MB/s eta 0:00:05
   -------------- ------------------------- 13.1/36.7 MB 5.7 MB/s eta 0:00:05
   ---------------- ----------------------- 14.9/36.7 MB 5.8 MB/s eta 0:00:04
  


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
from scipy.stats import chi2_contingency

monthly_table = pd.crosstab(
    df["order_month"],
    df["late_delivery_risk"]
)

chi2, p_value, dof, expected = chi2_contingency(monthly_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)

Chi-square statistic: 16.938821085516473
p-value: 0.1096983165930315


### Insight

The late delivery rates are very similar across the different months, ranging from approximately 54% to 56%.

The chi-square test produced a p-value of 0.1097. Since this is greater than 0.05, the result is not statistically significant.

Therefore, the null hypothesis is not rejected. Based on this dataset, there is not enough evidence to suggest that the month an order was placed is related to late delivery risk.

Although there are small differences between the months, these differences are not large enough to be considered statistically significant.

## Hypothesis 3: Certain product categories have higher late delivery rates than others

This hypothesis looks at whether late delivery risk is different between product categories.

To investigate this, I will:

- Group orders by `category_name`.
- Calculate the percentage of late deliveries for each category.
- Compare the categories using a bar chart.
- Use a chi-square test to see if there is a significant relationship between product category and late delivery risk.

**Null hypothesis (H₀):** There is no relationship between product category and late delivery risk.

**Alternative hypothesis (H₁):** There is a relationship between product category and late delivery risk.

In [46]:
category_late = (
    df.groupby("category_name")["late_delivery_risk"]
      .mean()
      .mul(100)
      .reset_index(name="late_delivery_percentage")
      .sort_values("late_delivery_percentage", ascending=False)
)

category_late


,category_name,late_delivery_percentage
23,Golf Bags & Carts,68.852459
32,Lacrosse,60.058309
37,Pet Supplies,58.943089
8,Cameras,58.108108
41,Strength Training,57.657658
1,As Seen on TV!,57.352941
36,Music,57.142857
0,Accessories,56.966292
19,Fitness Accessories,56.957929
5,Books,56.543210


In [47]:
fig = px.bar(
    category_late,
    x="category_name",
    y="late_delivery_percentage",
    title="Late Delivery Risk by Product Category",
    labels={
        "category_name": "Product Category",
        "late_delivery_percentage": "Late Delivery Risk (%)"
    }
)

fig.update_layout(
    xaxis={"categoryorder": "total descending"}
)

fig.show()

In [48]:
category_table = pd.crosstab(
    df["category_name"],
    df["late_delivery_risk"]
)

chi2, p_value, dof, expected = chi2_contingency(category_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)

Chi-square statistic: 42.88922829155516
p-value: 0.7179808169070767


### Insight

The late delivery rates vary slightly between product categories.

The chi-square test produced a p-value of 0.7180. Since this is greater than 0.05, the result is not statistically significant.

Therefore, the null hypothesis is not rejected. Based on this dataset, there is not enough evidence to suggest that product category is related to late delivery risk.

Although some categories have slightly higher or lower late delivery rates, the differences are not large enough to be considered statistically significant.

## Hypothesis 4: Customer country is associated with late delivery risk

This hypothesis looks at whether late delivery risk differs between customer countries.

To investigate this, I will:

- Group orders by `customer_country`.
- Calculate the percentage of late deliveries for each country.
- Compare the results using a bar chart.
- Consider whether the differences provide useful evidence about delivery reliability.

In [49]:
country_late = (
    df.groupby("customer_country")["late_delivery_risk"]
      .mean()
      .mul(100)
      .reset_index(name="late_delivery_percentage")
      .sort_values("late_delivery_percentage", ascending=False)
)

country_late


,customer_country,late_delivery_percentage
0,EE. UU.,54.870171
1,Puerto Rico,54.763381


In [50]:
fig = px.bar(
    country_late,
    x="customer_country",
    y="late_delivery_percentage",
    title="Late Delivery Risk by Customer Country",
    labels={
        "customer_country": "Customer Country",
        "late_delivery_percentage": "Late Delivery Risk (%)"
    }
)

fig.show()

### Insight

The results show that the late delivery rates are almost identical between the two customer countries.

EE. UU. has a late delivery rate of 54.87%, while Puerto Rico has a rate of 54.76%. The difference is only 0.11 percentage points.

This suggests that customer country does not have a meaningful effect on late delivery risk in this dataset.

The dataset also has very limited country variation, with only two countries represented, so this analysis provides limited information about the effect of geography on delivery performance.


## Hypothesis 5: High-value orders are more likely to be delivered late

This hypothesis looks at whether high-value orders have a higher chance of late delivery than regular orders.

To investigate this, I will:

- Compare late delivery rates between high-value and regular orders.
- Visualise the results using a bar chart.
- Evaluate whether there is a meaningful difference between the two groups.


In [51]:
late_rates = (
    df.groupby("high_value_order")["late_delivery_risk"]
      .mean()
      .mul(100)
      .reset_index(name="late_delivery_percentage")
)

late_rates


,high_value_order,late_delivery_percentage
0,False,54.961411
1,True,54.695942


In [52]:
fig = px.bar(
    late_rates,
    x="high_value_order",
    y="late_delivery_percentage",
    title="Late Delivery Risk: High-Value vs Regular Orders",
    labels={
        "high_value_order": "High-Value Order",
        "late_delivery_percentage": "Late Delivery Risk (%)"
    }
)

fig.show()

### Insight

The results show that late delivery rates are very similar for regular and high-value orders.

Regular orders have a late delivery rate of 54.96%, while high-value orders have a rate of 54.70%. The difference is only 0.27 percentage points.

This suggests that high-value orders are not more likely to experience late delivery in this dataset.

Therefore, the hypothesis is not supported by the results. Order value does not appear to have a meaningful effect on late delivery risk.

# EDA Summary and Conclusion

The exploratory data analysis was used to investigate factors that may be related to late delivery risk in the DataCo Supply Chain dataset.

The analysis found that shipping mode showed clear differences in late delivery rates, with some shipping modes having a much higher rate than others.

The analysis of order month did not show a statistically significant relationship with late delivery risk. The chi-square test produced a p-value of 0.1097, which is greater than 0.05.

Product category also did not show a statistically significant relationship with late delivery risk. The chi-square test produced a p-value of 0.7180.

The customer country analysis showed almost identical late delivery rates between the two countries in the dataset. This provided limited evidence that customer country affects delivery performance.

High-value orders also had a very similar late delivery rate to regular orders, suggesting that order value does not have a meaningful effect on late delivery risk.

Overall, the EDA shows that some factors appear to have more influence on delivery performance than others. These findings will be used to help select features for the machine-learning stage of the project.

### Limitations

There are some limitations to the analysis. The dataset contains only two customer countries, limiting the ability to investigate geographical differences. The dataset is also a historical dataset, so the results may not represent current supply-chain performance.

The analysis identifies relationships within the dataset but does not prove that one variable directly causes late delivery.

### Next Step

The next stage of the project will use supervised machine learning to investigate whether late delivery risk can be predicted using information about orders, customers, products and shipping.